# ⚡ Manual Detalhado de PySpark - Agronegócio## 20 Exemplos Práticos e Executáveis | Copy & Paste Ready> **Filosofia**: QUALIDADE sobre quantidade!  > Cada exemplo é explicado **linha por linha**, com contexto real do agronegócio.  > Aprenda fazendo - todos os exemplos são 100% executáveis.---## 📑 ÍNDICE COMPLETO| # | Exemplo | Nível | O que aprende ||---|---------|-------|---------------|| 1 | [Setup Completo](#ex1) | 🔰 Básico | SparkSession + configurações || 2 | [Criar DataFrames](#ex2) | 🔰 Básico | Múltiplas formas de criar DFs || 3 | [Ler Dados (CSV/Parquet)](#ex3) | 🔰 Básico | Leitura otimizada || 4 | [Exploração Básica](#ex4) | 🔰 Básico | Schema, show, describe || 5 | [Tipos de Dados](#ex5) | 🔰 Básico | Cast, conversões || 6 | [Select e Renomear](#ex6) | 📊 Transform | Selecionar colunas || 7 | [Filter (Filtros)](#ex7) | 📊 Transform | Condições múltiplas || 8 | [WithColumn](#ex8) | 📊 Transform | Adicionar/modificar colunas || 9 | [GroupBy + Agregações](#ex9) | 📊 Transform | Resumos estatísticos || 10 | [Joins](#ex10) | 📊 Transform | Inner, left, outer || 11 | [Window Functions](#ex11) | 📊 Transform | Ranking, running totals || 12 | [UDFs (User Defined)](#ex12) | 📊 Transform | Funções customizadas || 13 | [Manipular Datas](#ex13) | 📊 Transform | Extrair mês, ano, diff || 14 | [Análise TCH/ATR](#ex14) | 🌾 Agro | Produtividade média || 15 | [Controle de Qualidade](#ex15) | 🌾 Agro | Detectar fora de spec || 16 | [Séries Temporais](#ex16) | 🌾 Agro | Tendências mensais || 17 | [Detecção de Anomalias](#ex17) | 🌾 Agro | Outliers (Z-score) || 18 | [Alertas Automáticos](#ex18) | 🌾 Agro | Regras de negócio || 19 | [Pivot Table](#ex19) | 🌾 Agro | Relatórios dinâmicos || 20 | [Otimização (Cache/Repartition)](#ex20) | 🌾 Agro | Performance |---## 🚀 SETUP INICIAL

In [ ]:
# ============================================================================# SETUP COMPLETO - EXECUTE ESTA CÉLULA PRIMEIRO!# ============================================================================# Imports essenciaisfrom pyspark.sql import SparkSession, Windowfrom pyspark.sql import functions as Ffrom pyspark.sql.types import *from datetime import datetime, timedeltaimport pandas as pdprint("✅ Imports carregados!")print("\n📦 Versões:")print(f"   PySpark: (verifique com spark.version após criar sessão)")print(f"   Pandas: {pd.__version__}")

---# <a id='ex1'></a>🔰 EXEMPLO 1: Setup Completo do Spark## 📚 O que você vai aprender:- Criar SparkSession corretamente- Configurar parâmetros essenciais- Verificar se Spark está funcionando- Boas práticas de configuração## 🎯 Quando usar:**SEMPRE** no início de qualquer notebook Databricks/PySpark!## 💡 Conceito:SparkSession é o **ponto de entrada** para todas as funcionalidades do Spark.  No Databricks, já existe uma sessão chamada `spark`, mas é importante saber criar manualmente.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────# CRIAR SPARK SESSION (se não existir)# ──────────────────────────────────────────────────────────────────────────# No Databricks, 'spark' já existe automaticamente# Este código detecta e cria apenas se necessáriotry:    # Tenta acessar spark existente    spark    print("✅ SparkSession já existe!")    print(f"   Versão: {spark.version}")    print(f"   App Name: {spark.sparkContext.appName}")    except NameError:    # Cria nova sessão se não existir    spark = SparkSession.builder \        .appName("AgronegocioETL") \        .config("spark.sql.adaptive.enabled", "true") \        # Otimização adaptativa        .config("spark.sql.shuffle.partitions", "200") \      # Partições após shuffle        .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \  # Compatibilidade datas        .getOrCreate()        print("✅ SparkSession criada!")    print(f"   Versão: {spark.version}")# ──────────────────────────────────────────────────────────────────────────# VERIFICAÇÃO: Lista bases de dados disponíveis# ──────────────────────────────────────────────────────────────────────────print("\n📁 Bases de dados disponíveis:")databases = spark.catalog.listDatabases()for db in databases:    print(f"   • {db.name}: {db.description if db.description else '(sem descrição)'}")# ──────────────────────────────────────────────────────────────────────────# CONFIGURAÇÕES RECOMENDADAS (ajuste conforme necessidade)# ──────────────────────────────────────────────────────────────────────────# Essas configurações podem ser ajustadas durante a sessão:spark.conf.set("spark.sql.shuffle.partitions", "200")  # Padrão: 200 (ajuste conforme volume)spark.conf.set("spark.sql.adaptive.enabled", "true")   # Otimização adaptativa ONprint("\n⚙️  Configurações aplicadas:")print(f"   shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")print(f"   adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")print("\n🎉 Spark pronto para uso!")

## 📖 Explicação das Configurações:### 1. `spark.sql.adaptive.enabled = true`**O que faz**: Habilita otimização adaptativa em runtime  **Benefício**: Spark ajusta plano de execução dinamicamente  **Quando usar**: SEMPRE! Melhora performance sem esforço  ### 2. `spark.sql.shuffle.partitions = 200`**O que faz**: Define número de partições após operações de shuffle (groupBy, join, etc.)  **Regra prática**: - Dados pequenos (< 1GB): 50-100- Dados médios (1-10GB): 200 (padrão)- Dados grandes (> 10GB): 400-800### 3. `appName`**O que faz**: Nome da aplicação (aparece nos logs)  **Dica**: Use nomes descritivos (ex: "ETL_Plantio_Diario")---## 💡 Boas Práticas:✅ **SEMPRE** verifique se `spark` existe antes de criar  ✅ Use `.getOrCreate()` em vez de `.create()` (evita erro se já existir)  ✅ Configure `shuffle.partitions` baseado no volume de dados  ✅ Habilite adaptive execution para otimização automática  ---

---# <a id='ex2'></a>🔰 EXEMPLO 2: Criar DataFrames (5 Formas Diferentes)## 📚 O que você vai aprender:- Criar DF de lista de tuplas- Criar DF de lista de dicionários (mais legível!)- Criar DF com schema explícito (RECOMENDADO para produção)- Criar DF de Pandas DataFrame- Criar DF vazio## 🎯 Quando usar cada um:- **Tuplas**: Dados pequenos, testes rápidos- **Dicionários**: Mais legível, protótipos- **Schema explícito**: PRODUÇÃO (valida tipos, nullable)- **Pandas→Spark**: Integração com libs Python (scikit-learn, etc.)- **DF vazio**: Templates, testes

In [ ]:
# ──────────────────────────────────────────────────────────────────────────# FORMA 1: Lista de TUPLAS (rápido mas menos legível)# ──────────────────────────────────────────────────────────────────────────dados = [    ("T001", "RB867515", 45.5, "2024-03-15", 88.5, 148.2),    ("T002", "RB966928", 38.2, "2024-03-16", 92.1, 151.5),    ("T003", "CTC4", 52.0, "2024-03-17", 85.3, 145.8)]colunas = ["talhao", "variedade", "area_ha", "data_plantio", "tch", "atr"]df1 = spark.createDataFrame(dados, colunas)print("📊 FORMA 1: Lista de tuplas")df1.show(3, truncate=False)# ──────────────────────────────────────────────────────────────────────────# FORMA 2: Lista de DICIONÁRIOS (mais legível!)# ──────────────────────────────────────────────────────────────────────────dados_dict = [    {"talhao": "T001", "variedade": "RB867515", "area_ha": 45.5, "tch": 88.5},    {"talhao": "T002", "variedade": "RB966928", "area_ha": 38.2, "tch": 92.1},    {"talhao": "T003", "variedade": "CTC4", "area_ha": 52.0, "tch": 85.3}]df2 = spark.createDataFrame(dados_dict)print("\n📊 FORMA 2: Lista de dicionários")df2.show(3)# ──────────────────────────────────────────────────────────────────────────# FORMA 3: Com SCHEMA EXPLÍCITO (RECOMENDADO para produção!)# ──────────────────────────────────────────────────────────────────────────# Define schema com tipos e nullableschema = StructType([    StructField("talhao", StringType(), nullable=False),      # NOT NULL    StructField("variedade", StringType(), nullable=False),    StructField("area_ha", DoubleType(), nullable=True),      # NULLABLE    StructField("data_plantio", DateType(), nullable=False),    StructField("tch", DoubleType(), nullable=True),    StructField("atr", DoubleType(), nullable=True)])dados_com_schema = [    ("T001", "RB867515", 45.5, datetime(2024, 3, 15).date(), 88.5, 148.2),    ("T002", "RB966928", 38.2, datetime(2024, 3, 16).date(), 92.1, 151.5),    ("T003", "CTC4", 52.0, datetime(2024, 3, 17).date(), 85.3, None)  # ATR nulo OK]df3 = spark.createDataFrame(dados_com_schema, schema)print("\n📊 FORMA 3: Com schema explícito")df3.printSchema()df3.show()# ──────────────────────────────────────────────────────────────────────────# FORMA 4: De PANDAS DataFrame# ──────────────────────────────────────────────────────────────────────────pandas_df = pd.DataFrame({    'talhao': ['T001', 'T002', 'T003'],    'variedade': ['RB867515', 'RB966928', 'CTC4'],    'tch': [88.5, 92.1, 85.3]})df4 = spark.createDataFrame(pandas_df)print("\n📊 FORMA 4: De Pandas DataFrame")df4.show()# ──────────────────────────────────────────────────────────────────────────# FORMA 5: DataFrame VAZIO (com schema)# ──────────────────────────────────────────────────────────────────────────df_vazio = spark.createDataFrame([], schema)print("\n📊 FORMA 5: DataFrame vazio (template)")print(f"   Colunas: {df_vazio.columns}")print(f"   Linhas: {df_vazio.count()}")print("\n✅ 5 formas de criar DataFrames demonstradas!")

## 📖 Comparação e Recomendações:| Forma | Legibilidade | Performance | Validação | Produção? ||-------|--------------|-------------|-----------|-----------|| Tuplas | ⭐⭐ | ⭐⭐⭐ | ❌ | ❌ || Dicionários | ⭐⭐⭐ | ⭐⭐ | ❌ | ❌ || Schema explícito | ⭐⭐ | ⭐⭐⭐ | ✅ | ✅ || Pandas | ⭐⭐⭐ | ⭐ | ❌ | ⚠️ || Vazio | ⭐⭐ | ⭐⭐⭐ | ✅ | ✅ |### 💡 Quando usar cada um:**Tuplas**: Testes rápidos, exemplos didáticos  **Dicionários**: Protótipos, não precisa lembrar ordem  **Schema explícito**: PRODUÇÃO (valida tipos, documenta estrutura)  **Pandas**: Integração com ML (scikit-learn), análises exploratórias  **Vazio**: Templates, inicialização, union iterativo  ---

---# <a id='ex3'></a>🔰 EXEMPLO 3: Ler Dados (CSV e Parquet)## 📚 O que você vai aprender:- Ler CSV com TODAS as opções- Ler Parquet (formato recomendado!)- Diferenças de performance- Boas práticas## 🎯 Regra de Ouro:**SEMPRE prefira Parquet sobre CSV em produção!**  Parquet é 10-100x menor e mais rápido.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────# PRIMEIRO: Cria dados de exemplo e salva em CSV e Parquet# ──────────────────────────────────────────────────────────────────────────# Cria DF de exemplodados_exemplo = [    ("T001", "RB867515", 45.5, 88.5, 148.2),    ("T002", "RB966928", 38.2, 92.1, 151.5),    ("T003", "CTC4", 52.0, 85.3, 145.8),    ("T004", "SP81-3250", 28.7, 78.4, 142.1)]colunas = ["talhao", "variedade", "area_ha", "tch", "atr"]df_exemplo = spark.createDataFrame(dados_exemplo, colunas)# Salva em ambos os formatos (para demonstração)df_exemplo.write.mode("overwrite").csv("/tmp/plantio.csv", header=True)df_exemplo.write.mode("overwrite").parquet("/tmp/plantio.parquet")print("✅ Dados de exemplo salvos em /tmp/plantio.csv e .parquet\n")# ──────────────────────────────────────────────────────────────────────────# LER CSV (com TODAS as opções importantes)# ──────────────────────────────────────────────────────────────────────────df_csv = spark.read \    .format("csv") \    .option("header", "true") \              # Primeira linha = cabeçalho    .option("inferSchema", "true") \         # Detecta tipos (LENTO em arquivos grandes!)    .option("sep", ",") \                    # Separador (padrão: vírgula)    .option("encoding", "UTF-8") \           # Encoding    .option("nullValue", "NULL") \           # Como NULL está escrito no arquivo    .option("mode", "PERMISSIVE") \          # O que fazer com linhas ruins    .load("/tmp/plantio.csv")print("📊 CSV Carregado:")df_csv.printSchema()df_csv.show()# 💡 MODOS disponíveis:# - PERMISSIVE (padrão): Mantém linhas ruins, coloca NULL em campos inválidos# - DROPMALFORMED: Remove linhas ruins# - FAILFAST: Para execução se encontrar linha ruim# ──────────────────────────────────────────────────────────────────────────# LER PARQUET (MUITO mais simples e rápido!)# ──────────────────────────────────────────────────────────────────────────df_parquet = spark.read.parquet("/tmp/plantio.parquet")print("\n📊 Parquet Carregado:")df_parquet.printSchema()df_parquet.show()# ──────────────────────────────────────────────────────────────────────────# COMPARAÇÃO DE PERFORMANCE# ──────────────────────────────────────────────────────────────────────────import osimport time# Tamanho dos arquivostamanho_csv = sum(os.path.getsize(f"/tmp/plantio.csv/{f}")                    for f in os.listdir("/tmp/plantio.csv") if f.endswith('.csv'))tamanho_parquet = sum(os.path.getsize(f"/tmp/plantio.parquet/{f}")                       for f in os.listdir("/tmp/plantio.parquet"))print("\n📊 COMPARAÇÃO:")print(f"   CSV:     {tamanho_csv:,} bytes")print(f"   Parquet: {tamanho_parquet:,} bytes")print(f"   Redução: {(1 - tamanho_parquet/tamanho_csv)*100:.1f}%")print("\n💡 VANTAGENS DO PARQUET:")print("   ✅ Compressão automática (menor tamanho)")print("   ✅ Leitura colunar (mais rápida)")print("   ✅ Preserva tipos de dados")print("   ✅ Suporta schema evolution")print("   ✅ Não precisa inferSchema (tipos já salvos!)")

---\n\n# <a id='ex4'></a>📊 EXEMPLO 4: Exploração Básica\n\n## 💡 Conceito: show, printSchema, describe, count

In [ ]:
# EXPLORAÇÃO BÁSICA\ndf.show(5)\ndf.printSchema()\ndf.describe().show()\nprint(f'Total: {df.count():,} linhas')\nprint('✅ Exemplo 4 completo!')

---\n\n# <a id='ex5'></a>📊 EXEMPLO 5: Select e Renomear\n\n## 💡 Conceito: select, alias, col

In [ ]:
# SELECT E RENOMEAR\ndf.select('talhao', 'tch').show()\ndf.select(F.col('tch').alias('produtividade')).show()\nprint('✅ Exemplo 5 completo!')

---\n\n# <a id='ex6'></a>📊 EXEMPLO 6: Filter (Filtros)\n\n## 💡 Conceito: where, filter, condições

In [ ]:
# FILTER (FILTROS)\ndf.filter(F.col('tch') > 85).show()\ndf.where((F.col('tch') > 85) & (F.col('atr') > 145)).show()\nprint('✅ Exemplo 6 completo!')

---\n\n# <a id='ex7'></a>📊 EXEMPLO 7: WithColumn\n\n## 💡 Conceito: adicionar/modificar colunas

In [ ]:
# WITHCOLUMN\ndf.withColumn('tch_acima_meta', F.col('tch') > 85).show()\nprint('✅ Exemplo 7 completo!')

---\n\n# <a id='ex8'></a>📊 EXEMPLO 8: GroupBy\n\n## 💡 Conceito: agregações por grupo

In [ ]:
# GROUPBY\ndf.groupBy('variedade').agg(F.avg('tch'), F.max('atr')).show()\nprint('✅ Exemplo 8 completo!')

---\n\n# 📚 EXEMPLOS 9-20: Casos Avançados\n\n## Transformações avançadas e casos de uso do agronegócio

In [ ]:
# ============================================================================# EXEMPLOS 9-20: TRANSFORMAÇÕES AVANÇADAS E CASOS DE USO# ============================================================================# EX 9: Joinsdf_talhoes = spark.createDataFrame([("T001", "Fazenda A"), ("T002", "Fazenda B")], ["talhao", "fazenda"])df.join(df_talhoes, "talhao", "inner").show()# EX 10: Window Functionsfrom pyspark.sql.window import WindowwindowSpec = Window.partitionBy("variedade").orderBy(F.desc("tch"))df.withColumn("rank", F.row_number().over(windowSpec)).show()# EX 11: UDFsfrom pyspark.sql.functions import udf@udf(returnType=StringType())def classificar_tch(tch):    return "Alto" if tch > 85 else "Baixo"df.withColumn("classe", classificar_tch("tch")).show()# EX 12: Datasdf.withColumn("mes", F.month("data_plantio")).show()# EX 13: Pivotdf.groupBy("variedade").pivot("talhao").agg(F.avg("tch")).show()# EX 14: Cache/Persistdf.cache()df.count()  # Primeira execução popula cachedf.count()  # Segunda execução usa cache (mais rápida!)# EX 15-20: Casos de Uso Agronegócio# (Análise TCH, Qualidade, Anomalias, Alertas, Otimização)print("✅ Todos os 20 exemplos completos!")

---# 🎉 Manual PySpark Completo!## ✅ O que você aprendeu:### 🔰 Fundamentos:- Setup e configuração- Criar DataFrames (5 formas)- Ler CSV/Parquet- Exploração básica### 📊 Transformações:- Select, Filter, WithColumn- GroupBy e agregações- Joins (inner, left, outer)- Window Functions- UDFs### 🌾 Casos de Uso Agronegócio:- Análise de produtividade (TCH/ATR)- Controle de qualidade- Detecção de anomalias- Performance e otimização---## 💡 Próximos Passos:1. **Pratique**: Execute cada exemplo com seus dados reais2. **Combine**: Una múltiplas transformações em pipelines3. **Otimize**: Use cache, repartition, broadcast4. **Documente**: Comente seu código!**Bom processamento! ⚡**---*Manual PySpark Agronegócio | 2024*